In [1]:
import glob
import numpy as np
import pandas as pd

from pathlib import Path

csv_enums = glob.glob("./dataset/*")

column_translation = {
    "지점": "Station",
    "일시": "Date/Time",
    "기온(°C)": "Temperature (°C)",
    "1분 강수량(mm)": "1-minute Precipitation (mm)",
    "강수유무(유무)": "Precipitation Presence (Presence/Absence)",
    "풍향(deg)": "Wind Direction (deg)",
    "풍속(m/s)": "Wind Speed (m/s)",
    "현지기압(hPa)": "Local Pressure (hPa)",
    "해면기압(hPa)": "Sea-level Pressure (hPa)",
    "습도(%)": "Humidity (%)",
    "일사(MJ/m^2)": "Solar Radiation (MJ/m^2)",
    "일조(Sec)": "Sunshine Duration (Sec)",
}

def detect_encoding(file_path):
    """
    Try common Korean encodings.

    CP949 is commonly used for Korean meteorological CSV files.
    """
    file_path = Path(file_path)

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp949",
        "euc-kr",
    ]

    for encoding in encodings:
        try:
            with open(file_path, "r", encoding=encoding) as f:
                f.read(10000)

            return encoding

        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Could not determine encoding for {file_path}"
    )

# Load and Translate
monthly_data = dict()
for csv_enum in csv_enums:
    # Find Proper encoding to load korean
    encoding = detect_encoding(csv_enum)

    # Load CSV
    loaded_df = pd.read_csv(
        csv_enum, encoding=encoding, dtype=str, keep_default_na=False
    )

    # Translate
    translated_df = loaded_df.rename(columns=column_translation)
    
    key_month_name = pd.to_datetime(translated_df["Date/Time"]).dt.month_name()[0]

    # Store and sort based on months
    monthly_data[key_month_name] = translated_df

In [2]:
print(monthly_data.keys())
print(monthly_data['April'].keys())

dict_keys(['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August'])
Index(['Station', 'Date/Time', 'Temperature (°C)',
       '1-minute Precipitation (mm)',
       'Precipitation Presence (Presence/Absence)', 'Wind Direction (deg)',
       'Wind Speed (m/s)', 'Local Pressure (hPa)', 'Sea-level Pressure (hPa)',
       'Humidity (%)', 'Solar Radiation (MJ/m^2)', 'Sunshine Duration (Sec)'],
      dtype='object')


In [3]:
monthly_data['April']['Solar Radiation (MJ/m^2)']

0         
1         
2         
3         
4         
        ..
43194     
43195     
43196     
43197     
43198     
Name: Solar Radiation (MJ/m^2), Length: 43199, dtype: object

In [4]:
monthly_data['April']

,Station,Date/Time,Temperature (°C),1-minute Precipitation (mm),Precipitation Presence (Presence/Absence),Wind Direction (deg),Wind Speed (m/s),Local Pressure (hPa),Sea-level Pressure (hPa),Humidity (%),Solar Radiation (MJ/m^2),Sunshine Duration (Sec)
0,939,2026-04-01 00:01,10.6,0,0,323.3,.8,1008.2,1014.2,78.3,,
1,939,2026-04-01 00:02,10.6,0,0,301,.7,1008.2,1014.2,78.5,,
2,939,2026-04-01 00:03,10.6,0,0,303.8,.2,1008.3,1014.3,78.9,,
3,939,2026-04-01 00:04,10.5,0,0,306.7,1.7,1008.3,1014.3,79.1,,
4,939,2026-04-01 00:05,10.6,0,0,334.4,2.1,1008.3,1014.3,79,,
...,...,...,...,...,...,...,...,...,...,...,...,...
43194,939,2026-04-30 23:56,11.7,0,10,288.3,.5,1004,1009.9,93.1,,
43195,939,2026-04-30 23:57,11.7,0,10,304.7,1.5,1004,1009.9,93.1,,
43196,939,2026-04-30 23:58,11.7,0,10,294.7,.7,1004,1009.9,93.2,,
43197,939,2026-04-30 23:59,11.7,0,10,287,.3,1004,1009.9,93.2,,


In [5]:
N_STATES = 5
INITIAL_TRAIN_DAYS = 5
TRAIN_INTERVAL_DAYS = 5
HORIZON = 5
N_ITER = 50

In [6]:
from modules.evaluations import calculate_metrics
from modules.model import get_hmm_features, train_hmm, learn_state_scores, evaluate_day
from modules.feature_process import prepare_data, create_features, create_future_label
from modules.logging import setup_logging

from modules.debug import inspect_precipitation
from modules.export import save_evaluation_json

c:\Users\andro\.conda\envs\weather_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import logging

output_dir, log_file = setup_logging(
    "./results/"
)
logging.info(
    "Starting HMM walk-forward experiment"
)

df = prepare_data(
    monthly_data
)
inspect_precipitation(df)
df = create_features(df)
df = create_future_label(
    df,
    horizon=HORIZON
)

df["_date"] = (
    df["Date/Time"]
    .dt.normalize()
)

dates = sorted(
    df["_date"]
    .drop_duplicates()
)

2026-08-09 18:59:51,574 | INFO | Starting HMM walk-forward experiment



=== PRECIPITATION CHECK ===

1-minute Precipitation (mm)
dtype: float64
NaN: 776
Unique: [0.0, 0.5, 1.0]
Non-zero: 825
Maximum: 1.0
Value counts:
1-minute Precipitation (mm)
0.0    315237
0.5       824
1.0         1
Name: count, dtype: int64

Precipitation Presence (Presence/Absence)
dtype: float64
NaN: 824
Unique: [0.0, 10.0]
Non-zero: 14123
Maximum: 10.0
Value counts:
Precipitation Presence (Presence/Absence)
0.0     301891
10.0     14123
Name: count, dtype: int64


In [8]:
from tqdm import tqdm

n_folds = (
    len(dates) - INITIAL_TRAIN_DAYS
    + TRAIN_INTERVAL_DAYS - 1
) // TRAIN_INTERVAL_DAYS

logging.info(
    f"Total observations: {len(df):,}"
)

logging.info(
    f"Total days: {len(dates)}"
)

logging.info(
    f"Total folds: {n_folds}"
)

logging.info(
    f"N_STATES={N_STATES}, "
    f"INITIAL_TRAIN_DAYS={INITIAL_TRAIN_DAYS}, "
    f"TRAIN_INTERVAL_DAYS={TRAIN_INTERVAL_DAYS}, "
    f"HORIZON={HORIZON}, "
    f"N_ITER={N_ITER}"
)
# ----------------------------------------
# Storage
# ----------------------------------------

all_results = []
metrics = []

previous_hmm = None

# ----------------------------------------
# Walk forward
# ----------------------------------------

train_end = INITIAL_TRAIN_DAYS
fold = 0

with tqdm(
    total=n_folds,
    desc="Walk-forward HMM",
    unit="fold"
) as pbar:
    while train_end < len(dates):
        fold += 1
        train_dates = dates[:train_end]

        eval_dates = dates[
            train_end:
            train_end + 1
        ]

        if not eval_dates:
            break

        # ------------------------------------------------
        # Progress description
        # ------------------------------------------------

        pbar.set_postfix_str(
            f"Train={train_dates[0]:%Y-%m-%d}"
            f"→{train_dates[-1]:%Y-%m-%d} | "
            f"Eval={eval_dates[0]:%Y-%m-%d}"
        )

        logging.info(
            f"Fold {fold}/{n_folds} | "
            f"Train: "
            f"{train_dates[0]:%Y-%m-%d}"
            f"→"
            f"{train_dates[-1]:%Y-%m-%d} | "
            f"Eval: "
            f"{eval_dates[0]:%Y-%m-%d}"
        )

        # ------------------------------------------------
        # Split
        # ------------------------------------------------

        train_df = df[
            df["_date"].isin(train_dates)
        ]

        #print(train_df)

        eval_df = df[
            df["_date"].isin(eval_dates)
        ]

        logging.info(
            f"Training rows: {len(train_df):,} | "
            f"Evaluation rows: {len(eval_df):,}"
        )

        # ------------------------------------------------
        # Train HMM
        # ------------------------------------------------

        hmm, scaler = train_hmm(
            train_df,
            previous_hmm=previous_hmm,
            n_states=N_STATES,
            n_iter=N_ITER,
        )

        logging.info(
            f"HMM trained | "
            f"iterations={hmm.monitor_.iter} | "
            f"converged={hmm.monitor_.converged}"
        )

        # ------------------------------------------------
        # State → rain mapping
        # ------------------------------------------------

        state_scores = learn_state_scores(
            train_df,
            hmm,
            scaler,
        )

        logging.info(
            "State rain scores: "
            + np.array2string(
                state_scores,
                precision=4
            )
        )

        # ------------------------------------------------
        # Evaluate
        # ------------------------------------------------

        result = evaluate_day(
            eval_df,
            hmm,
            scaler,
            state_scores,
        )

        result["Fold"] = fold
        result["Eval_Date"] = eval_dates[0]

        all_results.append(result)

        # ------------------------------------------------
        # Metrics
        # ------------------------------------------------

        fold_metrics = calculate_metrics(
            result
        )

        eval_start = eval_df["Date/Time"].min()
        eval_end = eval_df["Date/Time"].max()

        fold_metrics.update({
            "Fold": fold,
            "Eval_Date": eval_dates[0],
            "Train_Start": train_dates[0],
            "Train_End": train_dates[-1],
            "N_Train_Rows": len(train_df),
            "N_Eval_Rows": len(result),
        })

        metrics.append(
            fold_metrics
        )

        # ------------------------------------------------
        # Save metrics immediately
        # ------------------------------------------------

        metrics_df = pd.DataFrame(
            metrics
        )

        print(fold)
        save_evaluation_json(
            result=result,
            fold=fold,
            train_start=train_dates[0],
            train_end=train_dates[-1],
            eval_start=eval_start,
            eval_end=eval_end,
            metrics=metrics_df,
            output_dir="evaluation_logs",
        )

        metrics_df.to_csv(
            output_dir /
            "fold_metrics.csv",
            index=False
        )

        # ------------------------------------------------
        # Save current evaluation
        # ------------------------------------------------

        result.to_csv(
            output_dir /
            f"fold_{fold:03d}_evaluation.csv",
            index=False
        )

        # ------------------------------------------------
        # Log metrics
        # ------------------------------------------------

        logging.info(
            f"Fold {fold} metrics | "
            f"Brier={fold_metrics['Brier']:.4f} | "
            f"ROC-AUC={fold_metrics['ROC_AUC']:.4f} | "
            f"PR-AUC={fold_metrics['PR_AUC']:.4f} | "
            f"F1={fold_metrics['F1']:.4f}"
        )

        # ------------------------------------------------
        # Keep model
        # ------------------------------------------------

        previous_hmm = hmm

        # ------------------------------------------------
        # Advance by TRAIN_INTERVAL_DAYS
        # ------------------------------------------------

        #print(type(TRAIN_INTERVAL_DAYS), TRAIN_INTERVAL_DAYS)
        train_end += TRAIN_INTERVAL_DAYS

        pbar.update(1)

    # ========================================================
    # COMBINE RESULTS
    # ========================================================

    results_df = (
        pd.concat(
            all_results,
            ignore_index=True
        )
        if all_results
        else pd.DataFrame()
    )

    metrics_df = pd.DataFrame(
        metrics
    )

    results_df.to_csv(
        output_dir /
        "all_evaluation_results.csv",
        index=False
    )

    metrics_df.to_csv(
        output_dir /
        "fold_metrics.csv",
        index=False
    )

    if len(results_df) > 0:

        overall = calculate_metrics(
            results_df
        )

        overall_df = pd.DataFrame(
            [overall]
        )

        overall_df.to_csv(
            output_dir /
            "overall_metrics.csv",
            index=False
        )

        logging.info(
            f"Overall metrics | "
            f"Brier={overall['Brier']:.4f} | "
            f"ROC-AUC={overall['ROC_AUC']:.4f} | "
            f"PR-AUC={overall['PR_AUC']:.4f} | "
            f"F1={overall['F1']:.4f}"
        )

    logging.info(
        f"Experiment complete. "
        f"Results saved to: {output_dir}"
    )

2026-08-09 18:59:53,143 | INFO | Total observations: 316,838
2026-08-09 18:59:53,144 | INFO | Total days: 221
2026-08-09 18:59:53,144 | INFO | Total folds: 44
2026-08-09 18:59:53,145 | INFO | N_STATES=5, INITIAL_TRAIN_DAYS=5, TRAIN_INTERVAL_DAYS=5, HORIZON=5, N_ITER=50
Walk-forward HMM:   0%|          | 0/44 [00:00<?, ?fold/s, Train=2026-01-01→2026-01-05 | Eval=2026-01-06]2026-08-09 18:59:53,148 | INFO | Fold 1/44 | Train: 2026-01-01→2026-01-05 | Eval: 2026-01-06
2026-08-09 18:59:53,153 | INFO | Training rows: 7,198 | Evaluation rows: 1,440
2026-08-09 18:59:55,265 | INFO | HMM trained | iterations=38 | converged=True
2026-08-09 18:59:55,277 | INFO | State rain scores: [0. 0. 0. 0. 0.]
2026-08-09 18:59:55,510 | INFO | Fold 1 metrics | Brier=0.0000 | ROC-AUC=0.0000 | PR-AUC=0.0000 | F1=0.0000


1


Walk-forward HMM:   2%|▏         | 1/44 [00:02<01:41,  2.36s/fold, Train=2026-01-01→2026-01-10 | Eval=2026-01-11]2026-08-09 18:59:55,512 | INFO | Fold 2/44 | Train: 2026-01-01→2026-01-10 | Eval: 2026-01-11
2026-08-09 18:59:55,518 | INFO | Training rows: 14,398 | Evaluation rows: 1,440
2026-08-09 18:59:56,335 | WARNING | Model is not converging.  Current: 94728.33821164069 is not greater than 94728.3415192677. Delta is -0.0033076270192395896
2026-08-09 18:59:56,336 | INFO | HMM trained | iterations=24 | converged=True
2026-08-09 18:59:56,350 | INFO | State rain scores: [0. 0. 0. 0. 0.]


2


2026-08-09 18:59:56,585 | INFO | Fold 2 metrics | Brier=0.0000 | ROC-AUC=0.0000 | PR-AUC=0.0000 | F1=0.0000
Walk-forward HMM:   5%|▍         | 2/44 [00:03<01:07,  1.61s/fold, Train=2026-01-01→2026-01-15 | Eval=2026-01-16]2026-08-09 18:59:56,587 | INFO | Fold 3/44 | Train: 2026-01-01→2026-01-15 | Eval: 2026-01-16
2026-08-09 18:59:56,594 | INFO | Training rows: 21,597 | Evaluation rows: 1,440
2026-08-09 18:59:57,832 | INFO | HMM trained | iterations=24 | converged=True
2026-08-09 18:59:57,854 | INFO | State rain scores: [0.0000e+00 0.0000e+00 5.5443e-04 0.0000e+00 6.7733e-01]


...

2026-08-09 19:08:44,901 | INFO | Fold 42 metrics | Brier=0.0000 | ROC-AUC=0.0000 | PR-AUC=0.0000 | F1=0.0000
Walk-forward HMM:  95%|█████████▌| 42/44 [08:51<00:36, 18.30s/fold, Train=2026-01-01→2026-08-03 | Eval=2026-08-04]2026-08-09 19:08:44,903 | INFO | Fold 43/44 | Train: 2026-01-01→2026-08-03 | Eval: 2026-08-04
2026-08-09 19:08:44,933 | INFO | Training rows: 309,578 | Evaluation rows: 1,440
2026-08-09 19:09:07,742 | INFO | HMM trained | iterations=37 | converged=True
2026-08-09 19:09:08,017 | INFO | State rain scores: [2.8049e-03 2.9923e-03 6.6872e-04 1.7995e-03 8.0928e-01]


43


2026-08-09 19:09:08,252 | INFO | Fold 43 metrics | Brier=0.0000 | ROC-AUC=0.0000 | PR-AUC=0.0000 | F1=0.0000
Walk-forward HMM:  98%|█████████▊| 43/44 [09:15<00:19, 19.81s/fold, Train=2026-01-01→2026-08-08 | Eval=2026-08-09]2026-08-09 19:09:08,254 | INFO | Fold 44/44 | Train: 2026-01-01→2026-08-08 | Eval: 2026-08-09
2026-08-09 19:09:08,282 | INFO | Training rows: 316,778 | Evaluation rows: 60
2026-08-09 19:09:30,435 | WARNING | Model is not converging.  Current: 2893357.782126338 is not greater than 2893357.783068799. Delta is -0.0009424607269465923
2026-08-09 19:09:30,447 | INFO | HMM trained | iterations=35 | converged=True
2026-08-09 19:09:30,759 | INFO | State rain scores: [2.7159e-03 2.9681e-03 6.5482e-04 1.7157e-03 8.0928e-01]
2026-08-09 19:09:30,789 | INFO | Fold 44 metrics | Brier=0.0000 | ROC-AUC=0.0000 | PR-AUC=0.0000 | F1=0.0000
Walk-forward HMM: 100%|██████████| 44/44 [09:37<00:00, 20.63s/fold, Train=2026-01-01→2026-08-08 | Eval=2026-08-09]

44


2026-08-09 19:09:32,080 | INFO | Overall metrics | Brier=0.0077 | ROC-AUC=0.9830 | PR-AUC=0.8514 | F1=0.8913
2026-08-09 19:09:32,081 | INFO | Experiment complete. Results saved to: results
Walk-forward HMM: 100%|██████████| 44/44 [09:38<00:00, 13.16s/fold, Train=2026-01-01→2026-08-08 | Eval=2026-08-09]


In [9]:
train_df

,Station,Date/Time,Temperature (°C),1-minute Precipitation (mm),Precipitation Presence (Presence/Absence),Wind Direction (deg),Wind Speed (m/s),Local Pressure (hPa),Sea-level Pressure (hPa),Humidity (%),Solar Radiation (MJ/m^2),Sunshine Duration (Sec),wind_dir_sin,wind_dir_cos,humidity_change_5min,pressure_change_5min,wind_change_5min,rain_5min,future_rain_score,_date
0,939,2026-01-01 00:01:00,-2.8,0.0,0.0,81.2,1.7,1017.5,1023.8,21.1,,,0.988228,0.152986,0.0,0.0,0.0,0.0,0.0,2026-01-01
1,939,2026-01-01 00:02:00,-2.8,0.0,0.0,88.1,2.2,1017.5,1023.8,21.1,,,0.999450,0.033155,0.0,0.0,0.0,0.0,0.0,2026-01-01
2,939,2026-01-01 00:03:00,-3.0,0.0,0.0,95.1,1.4,1017.5,1023.8,21.4,,,0.996041,-0.088894,0.0,0.0,0.0,0.0,0.0,2026-01-01
3,939,2026-01-01 00:04:00,-3.0,0.0,0.0,80.9,1.8,1017.4,1023.7,21.7,,,0.987414,0.158158,0.0,0.0,0.0,0.0,0.0,2026-01-01
4,939,2026-01-01 00:05:00,-2.9,0.0,0.0,66.4,0.9,1017.4,1023.7,21.8,,,0.916363,0.400349,0.0,0.0,0.0,0.0,0.0,2026-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316773,939,2026-08-08 23:55:00,28.1,0.0,0.0,52.3,2.7,1001.5,1007,65.7,,,0.791224,0.611527,0.1,0.0,0.4,0.0,0.0,2026-08-08
316774,939,2026-08-08 23:56:00,28.1,0.0,0.0,32.2,2.5,1001.5,1007,65.0,,,0.532876,0.846193,-0.6,0.0,0.7,0.0,0.0,2026-08-08
316775,939,2026-08-08 23:57:00,28.1,0.0,0.0,30.0,2.6,1001.5,1007,65.3,,,0.500000,0.866025,-0.4,0.0,-0.4,0.0,0.0,2026-08-08
316776,939,2026-08-08 23:58:00,28.1,0.0,0.0,50.5,4.0,1001.5,1007,64.8,,,0.771625,0.636078,-0.8,0.0,1.0,0.0,0.0,2026-08-08
